In [30]:
import pandas as pd
import os
from helpers import *
import openpyxl

In [31]:

path = "10991360_editet"

rows = []

for file in os.listdir(path):
    file_path = os.path.join(path, file)

    # Optional: process only DXF files
    if not file.lower().endswith(".dxf"):
        continue

    bounding_box_width, bounding_box_height = get_dxf_bounds(file_path)
    cutting_length = get_cutting_length(file_path)
    num_holes = get_number_of_holes(file_path)

    rows.append({
        "file": file,
        "bounding_box_width": bounding_box_width,
        "bounding_box_height": bounding_box_height,
        "cutting_length": cutting_length,
        "num_holes": num_holes,
    })

df = pd.DataFrame(rows)
# df.to_excel("ergebnisse.xlsx", index=False, engine="openpyxl")

In [32]:
# Bestelldaten einlesen
orders = pd.read_csv("customers_sales.csv", sep=None, engine="python")
orders.columns = orders.columns.str.strip()

for col in ["customer_name", "customer_branch", "file_name"]:
    orders[col] = orders[col].astype("string").str.strip()
orders["order_quantity"] = pd.to_numeric(orders["order_quantity"], errors="coerce")

# print(orders.shape)
# orders.head()

In [33]:
# Bestelldaten validieren
cols = ["customer_name", "customer_branch", "file_name", "order_quantity"]

print("Fehlende Werte:\n", orders[cols].isna().sum(), "\n")

invalid_qty = orders[
    orders["order_quantity"].isna()
    | (orders["order_quantity"] <= 0)
    | (orders["order_quantity"] % 1 != 0)
]
print("Ungültige Mengen:", len(invalid_qty))

exact_dups = orders[orders.duplicated(subset=cols, keep=False)]
print("Exakte Duplikate:", len(exact_dups))

# gleiche Kunde/Filiale/Datei, aber andere Menge (nur zur Prüfung, wird nicht gelöscht)
key = ["customer_name", "customer_branch", "file_name"]
same_order_diff_qty = orders[
    orders.duplicated(subset=key, keep=False) & ~orders.duplicated(subset=cols, keep=False)
]
print("Gleiche Order, andere Menge:", len(same_order_diff_qty))

Fehlende Werte:
 customer_name      0
customer_branch    0
file_name          0
order_quantity     0
dtype: int64 

Ungültige Mengen: 0
Exakte Duplikate: 8
Gleiche Order, andere Menge: 0


In [34]:
# DXF-Werte validieren
geo_cols = ["bounding_box_width", "bounding_box_height", "cutting_length"]

df["file"] = df["file"].astype("string").str.strip()

print("DXF-Dateien:", len(df))
print("Doppelte Dateinamen:", df["file"].duplicated().sum())
print("Fehlende Werte:\n", df[geo_cols + ["num_holes"]].isna().sum(), "\n")

invalid_geo = df[df[geo_cols].isna().any(axis=1) | (df[geo_cols] <= 0).any(axis=1)]
print("Fehlerhafte Geometriewerte:", len(invalid_geo))
invalid_geo

DXF-Dateien: 80
Doppelte Dateinamen: 0
Fehlende Werte:
 bounding_box_width     0
bounding_box_height    0
cutting_length         0
num_holes              0
dtype: int64 

Fehlerhafte Geometriewerte: 0


,file,bounding_box_width,bounding_box_height,cutting_length,num_holes


In [35]:
# Rohwerte der CSV (als Text), damit man bei nicht lesbaren Mengen sieht, was dort stand
raw = pd.read_csv("customers_sales.csv", sep=None, engine="python", dtype=str)
raw.columns = raw.columns.str.strip()

# --- 1) Gedroppt werden NUR exakte Duplikate (alle 4 Spalten gleich) ---
dup_mask = orders.duplicated(subset=cols, keep="first")

# Index der ersten Zeile, mit der das Duplikat übereinstimmt
first_idx = orders.assign(_idx=orders.index).groupby(cols, dropna=False)["_idx"].transform("min")

removed_report = orders[dup_mask].copy()
removed_report["grund"] = "Exaktes Duplikat (alle 4 Spalten gleich)"
removed_report["gleich_wie_index"] = first_idx[dup_mask]

print(f"Orders: {len(orders)} -> {len(orders) - len(removed_report)} "
      f"({len(removed_report)} entfernt)\n")
if len(removed_report):
    print("Entfernte Zeilen:")
    print(removed_report.to_string(), "\n")

orders_clean = orders[~dup_mask].copy()

# --- 2) Auffällige Zeilen: werden NICHT entfernt, nur angezeigt ---
suspicious_mask = (
    orders_clean[cols].isna().any(axis=1)
    | (orders_clean["order_quantity"] <= 0)
    | (orders_clean["order_quantity"] % 1 != 0)
)
suspicious = orders_clean[suspicious_mask].copy()
suspicious["rohwert_menge"] = raw.loc[suspicious.index, "order_quantity"]

print(f"Auffällige Zeilen (BEHALTEN, bitte prüfen): {len(suspicious)}")
if len(suspicious):
    print(suspicious.to_string())

# Menge nur in int umwandeln, wenn alle Werte gültig sind
if not suspicious_mask.any():
    orders_clean["order_quantity"] = orders_clean["order_quantity"].astype(int)

# --- 3) DXF-Werte: nur doppelte Dateinamen entfernen, ungültige Werte auf NaN setzen ---
dup_dxf = df[df.duplicated(subset="file", keep="first")]
print(f"\nEntfernte doppelte DXF-Zeilen: {len(dup_dxf)}")
if len(dup_dxf):
    print(dup_dxf.to_string())

df_clean = df.drop_duplicates(subset="file").copy()
for c in geo_cols:
    bad = ~(df_clean[c] > 0)
    if bad.any():
        print(f"\nUngültige Werte in {c} -> NaN gesetzt:")
        print(df_clean.loc[bad, ["file", c]].to_string())
    df_clean.loc[bad, c] = pd.NA

Orders: 79 -> 75 (4 entfernt)

Entfernte Zeilen:
        customer_name   customer_branch     file_name  order_quantity                                     grund  gleich_wie_index
7   PrecisionCut GmbH  Stuttgart Branch  Geo (15).DXF            4000  Exaktes Duplikat (alle 4 Spalten gleich)                 3
49       AutoParts AG     Munich Branch  Geo (55).DXF            9250  Exaktes Duplikat (alle 4 Spalten gleich)                42
62  PrecisionCut GmbH  Stuttgart Branch  Geo (56).DXF            4800  Exaktes Duplikat (alle 4 Spalten gleich)                50
68       AutoParts AG     Berlin Branch  Geo (12).DXF            8800  Exaktes Duplikat (alle 4 Spalten gleich)                 4 

Auffällige Zeilen (BEHALTEN, bitte prüfen): 0

Entfernte doppelte DXF-Zeilen: 0


In [36]:
# Zusammenführen
merged = orders_clean.merge(
    df_clean,
    left_on="file_name",
    right_on="file",
    how="left",
    validate="many_to_one",   # mehrere Orders pro DXF erlaubt, jede DXF nur einmal
    indicator=True,
).drop(columns="file")

print(merged["_merge"].value_counts())
print("Orders ohne DXF:", (merged["_merge"] == "left_only").sum())
print("Zeilen mit fehlenden DXF-Werten:", merged[geo_cols].isna().any(axis=1).sum())

merged = merged.drop(columns="_merge")
merged.head()

_merge
both          75
left_only      0
right_only     0
Name: count, dtype: int64
Orders ohne DXF: 0
Zeilen mit fehlenden DXF-Werten: 0


,customer_name,customer_branch,file_name,order_quantity,bounding_box_width,bounding_box_height,cutting_length,num_holes
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,1600.219048,305.140135,3438.024977,0
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,853.830000,333.401400,3217.207505,5
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,164.000000,145.120000,1742.517642,7
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,193.800891,138.442589,1302.774283,2
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,937.150697,74.006665,2764.783512,11


In [37]:
# DXF-Dateien ohne Bestellung (nur zur Info)
unused = set(df_clean["file"]) - set(orders_clean["file_name"])
print("DXF ohne Bestellung:", len(unused))

assert len(merged) == len(orders_clean), "Merge hat Zeilen verändert!"

# ---------- Tabellen der entfernten Zeilen vorbereiten ----------
orders_drop = removed_report.reset_index().rename(columns={"index": "original_index"})

dxf_drop = dup_dxf.reset_index().rename(columns={"index": "original_index"})
dxf_drop["grund"] = "Doppelter Dateiname (erste Zeile behalten)"

# ---------- Export ----------
with pd.ExcelWriter("customers_sales_with_dxf_data.xlsx", engine="openpyxl") as writer:
    merged.to_excel(writer, sheet_name="Daten", index=False)
    orders_drop.to_excel(writer, sheet_name="Entfernte Orders", index=False)
    dxf_drop.to_excel(writer, sheet_name="Entfernte DXF", index=False)

    # Spaltenbreiten automatisch anpassen
    for ws in writer.sheets.values():
        for column_cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in column_cells)
            ws.column_dimensions[column_cells[0].column_letter].width = min(width + 2, 60)

print(f"Export fertig: customers_sales_with_dxf_data.xlsx"
      f"({len(merged)} Orders, {len(orders_drop)} entfernte Orders, {len(dxf_drop)} entfernte DXF-Zeilen)")
merged

DXF ohne Bestellung: 5
Export fertig: customers_sales_with_dxf_data.xlsx(75 Orders, 4 entfernte Orders, 0 entfernte DXF-Zeilen)


,customer_name,customer_branch,file_name,order_quantity,bounding_box_width,bounding_box_height,cutting_length,num_holes
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,1600.219048,305.140135,3438.024977,0
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,853.830000,333.401400,3217.207505,5
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,164.000000,145.120000,1742.517642,7
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,193.800891,138.442589,1302.774283,2
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,937.150697,74.006665,2764.783512,11
...,...,...,...,...,...,...,...,...
70,AutoParts AG,Munich Branch,Geo (78).DXF,8150,386.361049,290.000000,5581.874650,52
71,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,36.553638,36.553638,147.032435,4
72,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,801.705747,344.180571,6111.748725,15
73,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,1968.901094,832.434782,16821.059941,0
